# Core functions

### Buld primary_id
Concatenates the configured primary key column(s) into a single `primary_id` string

In [0]:
def build_primary_id(df, primary_key_columns):

    primary_keys = [
        col.strip()
        for col in primary_key_columns.split(",")
    ]

    return df.withColumn(
        "primary_id",
        F.concat_ws(
            "||",
            *[
                F.col(pk).cast("string")
                for pk in primary_keys
            ]
        )
    )

### Current Version
Returns the latest committed Delta version for a table. Uses `DeltaTable.history(1)`

In [0]:
def get_current_version(full_table_name):

    delta_table = DeltaTable.forName(spark, full_table_name)

    current_version = (
        delta_table
        .history(1)
        .select("version")
        .first()["version"]
    )

    return current_version

### Process
Reads CDF for the unprocessed version range, extracts DELETE rows, and upserts them into the audit table.

In [0]:
def process_table(
    catalog_name,
    schema_name,
    table_name,
    primary_key_columns,
    last_processed_version
):

    full_table_name = (
        f"{catalog_name}.{schema_name}.{table_name}"
    )

    print("=" * 80)
    print(f"Processing table: {full_table_name}")
    print(f"Last processed version: {last_processed_version}")

    # Get current Delta version

    current_version = get_current_version(
        full_table_name
    )

    print(f"Current version: {current_version}")

    # Nothing new to process
    if current_version <= last_processed_version:

        print("No new Delta versions to process.")

        return current_version

    # Calculate CDF range

    start_version = last_processed_version + 1
    end_version = current_version

    print(
        f"Reading CDF versions "
        f"{start_version} -> {end_version}"
    )

    # Read CDF

    cdf_df = (
        spark.read
        .format("delta")
        .option("readChangeData", "true")
        .option("startingVersion", start_version)
        .option("endingVersion", end_version)
        .table(full_table_name)
    )


    # Keep only DELETE records

    delete_df = (
        cdf_df
        .filter(
            F.col("_change_type") == "delete"
        )
    )

    # Check whether DELETEs exist

    if delete_df.limit(1).count() == 0:

        print("No DELETE records found.")

        return current_version


    # Create primary_id

    delete_df = build_primary_id(
        delete_df,
        primary_key_columns
    )


    # Read Delta table history

    history_df = spark.sql(
        f"DESCRIBE HISTORY {full_table_name}"
    )


    # Keep only required columns

    history_df = history_df.select(
        F.col("version"),
        F.col("timestamp"),
        F.col("userId"),
        F.col("userName"),
        F.col("operation")
    )


    # Keep DELETE transactions only

    delete_history_df = (
        history_df
        .filter(
            F.upper(F.col("operation")) == "DELETE"
        )
        .filter(
            (F.col("version") >= start_version) &
            (F.col("version") <= end_version)
        )
    )

    # Join CDF with transaction history

    delete_df = delete_df.join(
        delete_history_df,
        delete_df["_commit_version"]
        == delete_history_df["version"],
        "left"
    )

    # Build audit records

    audit_df = delete_df.select(

        F.expr("uuid()").alias(
            "audit_id"
        ),

        F.lit(catalog_name).alias(
            "catalog_name"
        ),

        F.lit(schema_name).alias(
            "schema_name"
        ),

        F.lit(table_name).alias(
            "table_name"
        ),

        F.col("primary_id"),

        F.coalesce(
            F.col("userName"),
            F.col("userId")
        ).alias(
            "deleted_by"
        ),

        F.col("_commit_timestamp").alias(
            "deleted_at"
        ),

        F.col("_commit_version").alias(
            "commit_version"
        ),

        F.col("_commit_timestamp").alias(
            "commit_timestamp"
        ),

        F.lit("DELETE").alias(
            "operation"
        ),

        F.current_timestamp().alias(
            "loaded_at"
        )
    )

    # Upsert into audit log (idempotent write)

    audit_target = DeltaTable.forName(spark, AUDIT_TABLE)

    (
        audit_target.alias("target")
        .merge(
            audit_df.alias("source"),
            """
            target.table_name = source.table_name
            AND target.primary_id = source.primary_id
            AND target.commit_version = source.commit_version
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )


    print(
        f"Audit records written for "
        f"{full_table_name}"
    )

    # Return successfully processed version

    return current_version